In [13]:
import requests
import pandas as pd
import json

In [ ]:
APP_INSIGHTS_APP_ID = ""
API_KEY = ""

In [ ]:
# query = """
# traces 
# | where timestamp > ago(7d) 
# | where severityLevel == 3 
# | where tostring(customDimensions["meta_filename"]) == "doc.pdf"
# """

In [3]:
# Input variables
time_range = "1d"                      # e.g., "7d", or None
severity_level = 3                    # e.g., 3, or None
doc_name = "doc.pdf"                  # e.g., "doc.pdf", or None

In [4]:
# Start base query
query_parts = ["traces"]

# Time range
if time_range:
    query_parts.append(f"| where timestamp > ago({time_range})")

# Severity level
if severity_level is not None:
    query_parts.append(f"| where severityLevel == {severity_level}")

# Document name
if doc_name:
    query_parts.append(f'| where tostring(customDimensions["meta_filename"]) == "{doc_name}"')

# Final query string
query = "\n".join(query_parts)

print(query)

traces
| where timestamp > ago(1d)
| where severityLevel == 3
| where tostring(customDimensions["meta_filename"]) == "doc.pdf"


In [5]:
headers = {
    "x-api-key": API_KEY,
    "Content-Type": "application/json"
}

In [6]:
url = f"https://api.applicationinsights.io/v1/apps/{APP_INSIGHTS_APP_ID}/query"

In [7]:
response = requests.post(url, headers=headers, json={"query": query})

In [8]:
if response.status_code == 200:
    data = response.json()
    for row in data["tables"][0]["rows"]:
        print(row)
else:
    print(f"Failed: {response.status_code} - {response.text}")

['2025-08-07T10:25:07.772921Z', 'Document chunking started', 3, 'trace', '{"code.filepath":"C:\\\\Users\\\\jhagan.a\\\\Documents\\\\Tetratech\\\\Data ingestion\\\\document_processor_verbalization_chunking\\\\document_processor\\\\custom_logging_to_app_insights_v2.py","code.function":"log_step","code.lineno":"106","correlation_id":"xyz-456","step":"indexing","meta_filename":"doc.pdf"}', None, '', '4a15f9d1c2bba4c6f8ec776e7bc6c51d', '0e385befcd0b654f', '', '', '', '', '', '', 'PC', 'Other', 'Windows', '0.0.0.0', 'Mumbai', 'Maharashtra', 'India', 'Other', 'unknown_service', 'ILDCHNLAP0770', '23e47ac7-5c84-418c-aab6-0913ffdf1bb5', '/subscriptions/7288d743-5a3e-43c0-99ae-ba183840a669/resourcegroups/rg-contoso1/providers/microsoft.insights/components/appi-oddeuyvmhun6m', '92788bcd-6479-4d37-be2e-349dcda88112', 'uwm_py3.12.8:otel1.31.1:ext1.0.0b40', 'd613e47a-7378-11f0-8588-000d3ae3ea41', 1, '/subscriptions/7288d743-5a3e-43c0-99ae-ba183840a669/resourcegroups/rg-contoso1/providers/microsoft.in

In [9]:
# Parse and print as DataFrame
if response.status_code == 200:
    data = response.json()
    rows = data['tables'][0]['rows']
    columns = data['tables'][0]['columns']
    column_names = [col['name'] for col in columns]
    
    df = pd.DataFrame(rows, columns=column_names)

    # display(df)
    
    df["customDimensions"] = df["customDimensions"].apply(json.loads)

    custom_flat = pd.json_normalize(df["customDimensions"])

    # display(custom_flat)

    df_combined = pd.concat([df.drop(columns=["customDimensions"]), custom_flat], axis=1)

    display(df_combined)

else:
    print("Error:", response.status_code, response.text)

,timestamp,message,severityLevel,itemType,customMeasurements,operation_Name,operation_Id,operation_ParentId,operation_SyntheticSource,session_Id,...,sdkVersion,itemId,itemCount,_ResourceId,code.filepath,code.function,code.lineno,correlation_id,step,meta_filename
0,2025-08-07T10:25:07.772921Z,Document chunking started,3,trace,None,,4a15f9d1c2bba4c6f8ec776e7bc6c51d,0e385befcd0b654f,,,...,uwm_py3.12.8:otel1.31.1:ext1.0.0b40,d613e47a-7378-11f0-8588-000d3ae3ea41,1,/subscriptions/7288d743-5a3e-43c0-99ae-ba18384...,C:\Users\jhagan.a\Documents\Tetratech\Data ing...,log_step,106,xyz-456,indexing,doc.pdf
1,2025-08-07T05:32:53.047893Z,Document chunking started,3,trace,None,,30c7f368d866ae04bc25bfcf5356edc1,93cb72352e4d9f38,,,...,uwm_py3.12.8:otel1.31.1:ext1.0.0b40,0184096d-7350-11f0-8586-7c1e52b94444,1,/subscriptions/7288d743-5a3e-43c0-99ae-ba18384...,C:\Users\jhagan.a\Documents\Tetratech\Data ing...,log_step,105,xyz-456,indexing,doc.pdf


In [ ]:
import requests
import pandas as pd
import json

APP_INSIGHTS_APP_ID = ""
API_KEY = ""

In [2]:
# traces_query = """
# let startTime = datetime(2025-08-11T12:03:00Z);
# let endTime = datetime(2025-08-11T12:04:00Z);

# traces
# | where timestamp between (startTime .. endTime)
# """

def build_kql_queries(start_timestamp, end_timestamp):
    """
    Returns KQL queries for traces, dependencies, and exceptions for the given time range.
    """
    traces_query = f"""
        let startTime = datetime({start_timestamp});
        let endTime = datetime({end_timestamp});

        traces
        | where timestamp between (startTime .. endTime)
    """
    dependencies_query = f"""
        let startTime = datetime({start_timestamp});
        let endTime = datetime({end_timestamp});

        dependencies
        | where timestamp between (startTime .. endTime)
    """
    exceptions_query = f"""
        let startTime = datetime({start_timestamp});
        let endTime = datetime({end_timestamp});

        exceptions
        | where timestamp between (startTime .. endTime)
    """
    return traces_query, dependencies_query, exceptions_query

In [3]:
def run_kql_query(app_id: str, api_key: str, kql_query: str) -> pd.DataFrame:
    """
    Executes a KQL query on Azure Application Insights and returns the result as a DataFrame.
    Automatically flattens 'customDimensions' if present.

    Parameters:
        app_id (str): The Application Insights App ID.
        api_key (str): The API key with read permissions.
        kql_query (str): The KQL query string.

    Returns:
        pd.DataFrame: Resulting DataFrame, with flattened 'customDimensions' if applicable.
    """
    endpoint = f"https://api.applicationinsights.io/v1/apps/{app_id}/query"
    headers = {
        "x-api-key": api_key
    }
    params = {
        "query": kql_query
    }

    response = requests.get(endpoint, headers=headers, params=params)

    if response.status_code == 200:
        data = response.json()
        rows = data['tables'][0]['rows']
        columns = data['tables'][0]['columns']
        column_names = [col['name'] for col in columns]

        df = pd.DataFrame(rows, columns=column_names)

        return df

    else:
        raise Exception(f"Query failed: {response.status_code} {response.text}")

In [4]:
def get_last_child_info_until_http_or_blob(start_id, df):
    current_id = start_id
    visited = set()
    df_indexed = df.set_index("id")
    last_inproc_row = None

    while True:
        if current_id in visited:
            break  # Prevent circular loops
        visited.add(current_id)

        if current_id not in df_indexed.index:
            break

        row = df_indexed.loc[current_id]

        # Stop if type is "HTTP" or "Azure blob"
        if row["type"].lower() in ("https", "azure blob"):
            break

        if row["type"].lower() == "inproc":
            last_inproc_row = row

        # Move to first child
        children = df[df["operation_ParentId"] == current_id]
        if children.empty:
            break

        current_id = children.iloc[0]["id"]

    return last_inproc_row[["target", "type"]] if last_inproc_row is not None else None

In [5]:
def get_oldest_exception_with_dependency(
    exceptions_df, dependencies_df, start_time, end_time
    ):
    # Convert timestamp columns
    exceptions_df['timestamp'] = pd.to_datetime(exceptions_df['timestamp'])
    dependencies_df['timestamp'] = pd.to_datetime(dependencies_df['timestamp'])

    # Filter exceptions within the given time range
    filtered_exceptions = exceptions_df[
        (exceptions_df['timestamp'] >= start_time) &
        (exceptions_df['timestamp'] <= end_time)
    ]

    if filtered_exceptions.empty:
        return pd.DataFrame()  # No records in range

    # Find the oldest exception
    oldest_exception = filtered_exceptions.sort_values('timestamp').iloc[0:1]

    # Perform the join
    joined_df = pd.merge(
        oldest_exception,
        dependencies_df,
        how='left',
        left_on='operation_ParentId',
        right_on='id',
        # suffixes=('_exception_table', '_dependency_table')
    )

    return joined_df

In [6]:
def get_concatenated_messages(parent_id, traces_df, sep=" | "):
    # Filter matching rows
    matches = traces_df[traces_df["operation_ParentId"] == parent_id]

    if matches.empty:
        return None

    # Drop NaN messages, convert to string, and join
    concatenated = sep.join(matches["message"].dropna().astype(str))
    return concatenated

In [7]:
def get_final_df(output_df, columns_to_display, concatenated_message):
    # Filter columns
    final_df = output_df[columns_to_display].copy()

    # Apply the get_concatenated_messages function
    final_df["request_response_message"] = concatenated_message

    return final_df

In [8]:
def get_logs_for_time_range(start_timestamp, end_timestamp):

    traces_query, dependencies_query, exceptions_query = build_kql_queries(start_timestamp, end_timestamp)
    # Run KQL queries
    traces_df = run_kql_query(APP_INSIGHTS_APP_ID, API_KEY, traces_query)
    dependencies_df = run_kql_query(APP_INSIGHTS_APP_ID, API_KEY, dependencies_query)
    exceptions_df = run_kql_query(APP_INSIGHTS_APP_ID, API_KEY, exceptions_query)

    # Get oldest exception with dependency
    output_df = get_oldest_exception_with_dependency(
        exceptions_df, dependencies_df, pd.Timestamp(start_timestamp), pd.Timestamp(end_timestamp)
    )

    # Get last child info
    if not output_df.empty:
        parent_id = output_df["operation_ParentId_x"].iloc[0]
        last_child_info = get_last_child_info_until_http_or_blob(parent_id, dependencies_df)
        if last_child_info is not None:
            trace_parent_id = last_child_info.name
        else:
            trace_parent_id = parent_id
        # print(last_child_info.name)
        messages = get_concatenated_messages(trace_parent_id, traces_df, sep=" \n ")
        # print(messages)
    else:
        messages = None

    # Columns to display (reuse your list)
    columns_to_display = [
        'timestamp_x', 'type_x', 'outerMessage', 'details', 'customDimensions_x',
        'operation_Id_x', 'operation_ParentId_x', 'client_Type_x', 'client_OS_x',
        'client_City_x', 'client_StateOrProvince_x', 'client_CountryOrRegion_x',
        'cloud_RoleInstance_x', 'id', 'target', 'type_y', 'duration',
        'performanceBucket', 'operation_ParentId_y'
    ]

    # Build final_df
    final_df = get_final_df(output_df, columns_to_display, messages)

    final_df = final_df.rename(columns={
        "timestamp_x": "timestamp",
        "type_x": "exception_type",
        "outerMessage": "exception_message",
        "details": "exception_details",
        "operation_Id_x": "exception_operation_Id",
        "operation_ParentId_x": "exception_operation_ParentId_x",
        "client_Type_x": "client_Type",
        "client_OS_x": "client_OS",
        "client_City_x": "client_City",
        "client_StateOrProvince_x": "client_StateOrProvince",
        "client_CountryOrRegion_x": "client_CountryOrRegion",
        "cloud_RoleInstance_x": "cloud_RoleInstance",
        "id": "dependency_id",
        "target": "target_function",
        "type_y": "dependency_type",
        "operation_ParentId_y": "dependency_operation_ParentId",
    })
    return final_df

In [12]:
start_timestamp = '2025-08-22T13:15:00Z'
end_timestamp = '2025-08-22T13:16:00Z'

In [13]:
traces_query, dependencies_query, exceptions_query = build_kql_queries(start_timestamp, end_timestamp)

# Run KQL queries
traces_df = run_kql_query(APP_INSIGHTS_APP_ID, API_KEY, traces_query)
dependencies_df = run_kql_query(APP_INSIGHTS_APP_ID, API_KEY, dependencies_query)
exceptions_df = run_kql_query(APP_INSIGHTS_APP_ID, API_KEY, exceptions_query)

In [14]:
# Get oldest exception with dependency
output_df = get_oldest_exception_with_dependency(
    exceptions_df, dependencies_df, pd.Timestamp(start_timestamp), pd.Timestamp(end_timestamp)
)

In [15]:
display(output_df)

,timestamp_x,problemId,handledAt,type_x,message,assembly,method,outerType,outerMessage,outerAssembly,...,client_Browser_y,cloud_RoleName_y,cloud_RoleInstance_y,appId_y,appName_y,iKey_y,sdkVersion_y,itemId_y,itemCount_y,_ResourceId_y
0,2025-08-22 13:15:35.907835+00:00,HttpResponseError,,HttpResponseError,,Unknown,Unknown,HttpResponseError,Exception occurred: The specified resource nam...,,...,Other,unknown_service,ILDCHNLAP0770,23e47ac7-5c84-418c-aab6-0913ffdf1bb5,/subscriptions/7288d743-5a3e-43c0-99ae-ba18384...,92788bcd-6479-4d37-be2e-349dcda88112,uwm_py3.12.8:otel1.36.0:ext1.0.0b41,281fd18b-7f5a-11f0-a6fd-7c1e529571d9,1,/subscriptions/7288d743-5a3e-43c0-99ae-ba18384...


In [16]:
# Get last child info
if not output_df.empty:
    parent_id = output_df["operation_ParentId_x"].iloc[0]
    last_child_info = get_last_child_info_until_http_or_blob(parent_id, dependencies_df)
    if last_child_info is not None:
        trace_parent_id = last_child_info.name
    else:
        trace_parent_id = parent_id
    # print(last_child_info.name)
    messages = get_concatenated_messages(trace_parent_id, traces_df, sep=" \n ")
    # print(messages)
else:
    messages = None

In [18]:
print(last_child_info.name)

50dd18a02b5b09bd


In [16]:
# Columns to display (reuse your list)
columns_to_display = [
    'timestamp_x', 'type_x', 'outerMessage', 'details', 'customDimensions_x',
    'operation_Id_x', 'operation_ParentId_x', 'client_Type_x', 'client_OS_x',
    'client_City_x', 'client_StateOrProvince_x', 'client_CountryOrRegion_x',
    'cloud_RoleInstance_x', 'id', 'target', 'type_y', 'duration',
    'performanceBucket', 'operation_ParentId_y'
]

# Build final_df
final_df = get_final_df(output_df, columns_to_display, messages)

In [17]:
final_df = final_df.rename(columns={
    "timestamp_x": "timestamp",
    "type_x": "exception_type",
    "outerMessage": "exception_message",
    "details": "exception_details",
    "operation_Id_x": "exception_operation_Id",
    "operation_ParentId_x": "exception_operation_ParentId_x",
    "client_Type_x": "client_Type",
    "client_OS_x": "client_OS",
    "client_City_x": "client_City",
    "client_StateOrProvince_x": "client_StateOrProvince",
    "client_CountryOrRegion_x": "client_CountryOrRegion",
    "cloud_RoleInstance_x": "cloud_RoleInstance",
    "id": "dependency_id",
    "target": "target_function",
    "type_y": "dependency_type",
    "operation_ParentId_y": "dependency_operation_ParentId",
})

In [18]:
# Temporarily remove display truncation
with pd.option_context('display.max_colwidth', None):
    for col in final_df.columns:
        print(f"\n--- {col} ---")
        print(final_df[col].to_string(index=False))


--- timestamp ---
2025-08-11 12:03:50.326022+00:00

--- exception_type ---
ClientAuthenticationError

--- exception_message ---
Exception occurred: (401) Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.\nCode: 401\nMessage: Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.

--- exception_details ---
[{"severityLevel":"Error","outerId":"0","message":"Exception occurred: (401) Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.\nCode: 401\nMessage: Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and us

In [83]:
import pandas as pd

In [84]:
from query_app_ins_modular import get_logs_for_time_range

In [20]:
# Doc int error
df1 = get_logs_for_time_range('2025-08-11T12:03:00Z', '2025-08-11T12:04:00Z')

In [21]:
display(df1)

,timestamp,exception_type,exception_message,exception_details,customDimensions_x,exception_operation_Id,exception_operation_ParentId_x,client_Type,client_OS,client_City,client_StateOrProvince,client_CountryOrRegion,cloud_RoleInstance,dependency_id,target_function,dependency_type,duration,performanceBucket,dependency_operation_ParentId,request_response_message
0,2025-08-11 12:03:50.326022+00:00,ClientAuthenticationError,Exception occurred: (401) Access denied due to...,"[{""severityLevel"":""Error"",""outerId"":""0"",""messa...","{""user"":""abc123"",""code.file.path"":""C:\\Users\\...",ab60ec8498dc4c81002c86c6ab5002d9,dfdb265f55153ffe,PC,Windows,Bengaluru,Karnataka,India,ILDCHNLAP0770,dfdb265f55153ffe,analyze_document_fn,InProc,7419,7sec-15sec,1a1b372ace8f1786,Request URL: 'https://tetratech-doc-intelligen...


In [22]:
# Temporarily remove display truncation
with pd.option_context('display.max_colwidth', None):
    for col in df1.columns:
        print(f"\n--- {col} ---")
        print(df1[col].to_string(index=False))


--- timestamp ---
2025-08-11 12:03:50.326022+00:00

--- exception_type ---
ClientAuthenticationError

--- exception_message ---
Exception occurred: (401) Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.\nCode: 401\nMessage: Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.

--- exception_details ---
[{"severityLevel":"Error","outerId":"0","message":"Exception occurred: (401) Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.\nCode: 401\nMessage: Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and us

In [85]:
# openai error
df2 = get_logs_for_time_range('2025-08-12T06:27:00Z', '2025-08-12T06:29:00Z')

In [86]:
# Temporarily remove display truncation
with pd.option_context('display.max_colwidth', None):
    for col in df2.columns:
        print(f"\n--- {col} ---")
        print(df2[col].to_string(index=False))


--- timestamp ---
2025-08-12 06:28:30.726580+00:00

--- exception_type ---
AuthenticationError

--- exception_message ---
Exception occurred: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}

--- exception_details ---
[{"severityLevel":"Error","outerId":"0","message":"Exception occurred: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}","type":"AuthenticationError","id":"0","rawStack":"Traceback (most recent call last):\n  File \"C:\\Users\\jhagan.a\\Documents\\Tetratech\\Data ingestion\\document_processor_verbalization_chunking_with_enrich_metadata(upto_equipment)\\document_processor\\llm_metada

In [89]:
# Azure blob error
# df3 = get_logs_for_time_range('2025-08-14T11:06:00Z', '2025-08-14T11:07:00Z')
df3 = get_logs_for_time_range('2025-08-22T13:15:00Z', '2025-08-22T13:16:00Z')

In [90]:
# Temporarily remove display truncation
with pd.option_context('display.max_colwidth', None):
    for col in df3.columns:
        print(f"\n--- {col} ---")
        print(df3[col].to_string(index=False))


--- timestamp ---
2025-08-22 13:15:35.907835+00:00

--- exception_type ---
HttpResponseError

--- exception_message ---
Exception occurred: The specified resource name length is not within the permissible limits.\nRequestId:e7d57308-b01e-003a-0766-13e298000000\nTime:2025-08-22T13:15:35.1722596Z\nErrorCode:OutOfRangeInput\nContent: <?xml version="1.0" encoding="utf-8"?><Error><Code>OutOfRangeInput</Code><Message>The specified resource name length is not within the permissible limits.\nRequestId:e7d57308-b01e-003a-0766-13e298000000\nTime:2025-08-22T13:15:35.1722596Z</Message></Error>

--- exception_details ---
[{"severityLevel":"Error","outerId":"0","message":"Exception occurred: The specified resource name length is not within the permissible limits.\nRequestId:e7d57308-b01e-003a-0766-13e298000000\nTime:2025-08-22T13:15:35.1722596Z\nErrorCode:OutOfRangeInput\nContent: <?xml version=\"1.0\" encoding=\"utf-8\"?><Error><Code>OutOfRangeInput</Code><Message>The specified resource name lengt

In [68]:
# AI Search error
df4 = get_logs_for_time_range('2025-08-18T06:35:00Z', '2025-08-18T06:36:00Z')

In [69]:
# Temporarily remove display truncation
with pd.option_context('display.max_colwidth', None):
    for col in df4.columns:
        print(f"\n--- {col} ---")
        print(df4[col].to_string(index=False))


--- timestamp ---
2025-08-18 06:35:15.196551+00:00

--- exception_type ---
azure.core.exceptions.HttpResponseError

--- exception_message ---
() Request denied from Network Security Perimeter\nCode: \nMessage: Request denied from Network Security Perimeter

--- exception_details ---
[{"outerId":"0","message":"() Request denied from Network Security Perimeter\nCode: \nMessage: Request denied from Network Security Perimeter","type":"azure.core.exceptions.HttpResponseError","id":"0","rawStack":"Traceback (most recent call last):\n  File \"C:\\Users\\jhagan.a\\Documents\\Tetratech\\Data ingestion\\document_processor_verbalization_chunking_with_enrich_metadata(upto_equipment)\\document_processor\\venv\\Lib\\site-packages\\opentelemetry\\trace\\__init__.py\", line 589, in use_span\n    yield span\n  File \"C:\\Users\\jhagan.a\\Documents\\Tetratech\\Data ingestion\\document_processor_verbalization_chunking_with_enrich_metadata(upto_equipment)\\document_processor\\venv\\Lib\\site-packages\\op

In [ ]:
# Azure blob case - in case of multiple chain, list all the inproc till the last http or blob